In [ ]:
object_name = "yasone3"

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import astroalign

In [ ]:
from astropy.nddata import CCDData
from astropy.stats import mad_std

from ccdproc import ImageFileCollection
import ccdproc
from astropy.io import fits


In [ ]:
import astroalign

In [ ]:
import sys
sys.path.append("../")
sys.path.append("../../imaging/")

In [ ]:
from phot_utils import swap_byteorder

In [ ]:
from convenience_functions import show_image, show_images, show_image_residual

In [ ]:
import astroalign

In [ ]:
from pathlib import Path

In [ ]:
import arya
plt.rcParams["figure.dpi"] = 300

In [ ]:
import os

## Setup

In [ ]:
filters = ["g", "r", "i"]

In [ ]:
def open_image_file(obsname, band):
    if int(obsname) == 1:
        date_g = date_r = date_i = "20230708"
    elif int(obsname) == 2:
        date_g = date_r = "20230708"
        date_i = "20230709"
    elif int(obsname) == 3:
        date_g = date_r = date_i = "20230709"
    elif int(obsname) == 4:
        date_g = date_r = date_i = "20230709"
    elif int(obsname) == 5:
        date_g = date_r = "20230709"
        date_i = "20230710"
    elif int(obsname) == 6:
        date_g = date_r = date_i = "20230710"
    elif int(obsname) == 7:
        date_g = date_r = date_i = "20230724"
    else:
        date_g = date_r = date_i = "20230816"

    if band == "g":
        fits_file =  fits.open(f'../../imaging/julen_stacked//GTC73-23A_{obsname}_Sloan_g_{date_g}_NOSKY_BBI.fits')
    elif band == "r":
        fits_file =   fits.open(f'../../imaging/julen_stacked//GTC73-23A_{obsname}_Sloan_r_{date_r}_NOSKY_BBI.fits')
    elif band == "i":
        fits_file =   fits.open(f'../../imaging/julen_stacked//GTC73-23A_{obsname}_Sloan_i_{date_i}_NOSKY_BBI.fits')


    image_data = fits_file[0].data
    if image_data.dtype.byteorder not in ('=', '|'):
        image_data = image_data.astype(image_data.dtype.newbyteorder("="))


    return image_data, fits_file[0].header

In [ ]:
def get_image_files(obj, filt):
    if obj == "yasone1":
        obs = ["0001", "0002", "0003"]
    elif obj == "yasone2":
        obs = ["0004", "0005", "0006"]
    elif obj == "yasone3":
        obs = ["0007", "0008", "0009"]

        

    return [open_image_file(ob, filt)[0] for ob in obs]

In [ ]:
def get_header(obj, filt):
    if obj == "yasone1":
        obs = "0001"
    elif obj == "yasone2":
        obs = "0004"
    elif obj == "yasone3":
        obs = "0007"

    return open_image_file(obs, filt)[1]

In [ ]:
def align_all(images):
    image_0 = images[0]
    mask_combined = np.full_like(image_0, False, dtype=bool)

    images_aligned = [image_0]
    for i in range(1, len(images)):
        print(f"aligning image {i} / {len(images)-1}")
        image_aligned, mask = astroalign.register(images[i], image_0, max_control_points=60)
        mask_combined = mask_combined | mask
        images_aligned.append(image_aligned)

    return images_aligned, mask_combined

In [ ]:
imgs_obj = {filt: get_image_files(object_name, filt) for filt in filters}

In [ ]:
imgs_mask_obj_aligned = {filt: align_all(imgs) for filt, imgs in imgs_obj.items()}


In [ ]:

imgs_obj_aligned = {filt: imgs for filt, (imgs, mask) in imgs_mask_obj_aligned.items()}
mask_obj_aligned = {filt: mask for filt, (imgs, mask) in imgs_mask_obj_aligned.items()}


In [ ]:
def combine_images(filenames, **kwargs):
    return ccdproc.combine(filenames, 
                           method="sum",
                           sigma_clip=False, sigma_clip_low_thresh=5, sigma_clip_high_thresh=5, 
                           sigma_clip_func=np.ma.median, sigma_clip_dev_func=mad_std, 
                           mem_limit=1e9,
                           **kwargs
                            )

In [ ]:
imgs_obj_stacked = {filt: combine_images([CCDData(img, unit="adu", header=get_header(object_name, filt)) for img in imgs]) for filt, imgs in imgs_obj_aligned.items()}

In [ ]:
# for filt in filters:
#     imgs_obj_stacked[filt].mask = mask_obj_aligned[filt]


In [ ]:
imgs_obj_stacked["g"]

In [ ]:
imgs_obj_stacked["g"].write

In [ ]:
for filt in ["g", "r", "i"]:
    newpath = Path(f"../{object_name}/julen_stack_{filt}")
    if not newpath.is_dir():
        newpath.mkdir()
    
    imgs_obj_stacked[f"{filt}"].write(newpath / "coadd.fits", overwrite=True)


# Plots

The above image is an example of an alignment shift on an image

In [ ]:
for filt in filters:
    show_image(imgs_obj_stacked[filt], clabel="ADU", log=True)
    plt.title(filt)

## Save residual images

In [ ]:

show_images(imgs_obj_aligned[filters[1]])

In [ ]:
imgs_obj_stacked["g"].shape

In [ ]:
imgs_obj_stacked["g"].data - imgs_obj_aligned["g"][0]

In [ ]:
imgs_obj_aligned["g"][0].shape

In [ ]:
for filt in filters:
    for i in range(len(imgs_obj_aligned[filt])):
        show_image_residual(imgs_obj_aligned[filt][i], imgs_obj_stacked[filt].data)